# Speech emotion recognition: honest evaluation on RAVDESS

Runs every experiment for this repository on a free Colab GPU.

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all** (about 15–25 minutes)

The notebook downloads RAVDESS speech (about 200 MB, CC BY-NC-SA 4.0) into this Colab machine only.

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU"

In [ ]:
%cd /content
!rm -rf voice-emotion-detection
!git clone -q https://github.com/CODER7657/voice-emotion-detection.git
%cd /content/voice-emotion-detection
!pip -q install -e . 2>&1 | tail -2
# An editable install is only picked up by new processes, so make the
# already-running kernel see the package too.
import sys
sys.path.insert(0, "/content/voice-emotion-detection/src")

In [ ]:
import os
if not os.path.isdir("/content/ravdess/Actor_24"):
    !wget -q -O /content/ravdess.zip "https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1"
    !unzip -q -o /content/ravdess.zip -d /content/ravdess
from vemo.ravdess import build_manifest, describe
m = build_manifest("/content/ravdess")
print(describe(m))
assert len(m) == 1440 and m["actor"].nunique() == 24, "unexpected download"

## Run everything

Two tasks (`legacy3` = happy/sad/angry as in the original project, `all8` = every emotion),
four models, and two protocols (`leaky` = the original random split, `honest` = speaker-independent folds).

In [ ]:
!PYTHONPATH=src python scripts/run_experiments.py --data /content/ravdess --out results

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/results.md").read()))
for name in ["all8_wavlm_linear", "all8_cnn", "legacy3_cnn"]:
    path = f"results/confusion_{name}.png"
    if os.path.exists(path):
        display(Image(path, width=420))

## Compact results

Everything needed to rebuild the tables and plots locally, printed between two markers.

In [ ]:
import json
rows = json.load(open("results/results.json"))
def compact(r):
    res = r["result"]
    out = {"task": r["task"], "model": r["model"], "protocol": res["protocol"], "pooled": res["pooled"]}
    if res["protocol"] == "honest":
        out["macro_f1"] = res["macro_f1"]; out["accuracy"] = res["accuracy"]
        out["folds"] = [{"fold": f["fold"], "test_actors": f["test_actors"], "macro_f1": round(f["macro_f1"], 4),
                         "accuracy": round(f["accuracy"], 4)} for f in res["folds"]]
    return out
print("<<<VEMO_RESULTS>>>")
print(json.dumps([compact(r) for r in rows], separators=(",", ":")))
print("<<<END_VEMO_RESULTS>>>")